In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

In [5]:
df=pd.read_csv("../dataset/engineered/zomato_engineered.csv")

In [6]:
df.head()
df.shape
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41226 entries, 0 to 41225
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   name                         41226 non-null  str    
 1   cuisines                     41226 non-null  str    
 2   rest_type                    41226 non-null  str    
 3   location                     41226 non-null  str    
 4   rate                         41226 non-null  float64
 5   votes                        41226 non-null  int64  
 6   approx_cost(for two people)  41226 non-null  float64
 7   restaurant_profile           41226 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 2.5 MB


In [7]:
df["restaurant_profile"].head(10)

0    north indian, mughlai, chinese casual dining b...
1    chinese, north indian, thai casual dining bana...
2    cafe, mexican, italian cafe, casual dining ban...
3    south indian, north indian quick bites banasha...
4    north indian, rajasthani casual dining basavan...
5              north indian casual dining basavanagudi
6    north indian, south indian, andhra, chinese ca...
7    pizza, cafe, italian casual dining, cafe banas...
8         cafe, italian, continental cafe banashankari
9    cafe, mexican, italian, momos, beverages cafe ...
Name: restaurant_profile, dtype: str

In [8]:
tfidf=TfidfVectorizer(stop_words="english")

In [9]:
df.isnull().sum()

name                           0
cuisines                       0
rest_type                      0
location                       0
rate                           0
votes                          0
approx_cost(for two people)    0
restaurant_profile             0
dtype: int64

In [10]:
df.shape

(41226, 8)

In [11]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["restaurant_profile"])

In [12]:
tfidf_matrix.shape

(41226, 244)

In [13]:
type(tfidf_matrix)

scipy.sparse._csr.csr_matrix

In [17]:
from sklearn.metrics.pairwise import cosine_similarity
def recommend_restaurants(restaurant_name, top_n=10):
    if restaurant_name not in df["name"].values:
        return f"Restaurant '{restaurant_name}' not found."
    idx=df[df["name"]==restaurant_name].index[0]
    similarity_scores=cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    similar_indices=similarity_scores.argsort()[::-1]
    similar_indices=similar_indices[similar_indices != idx]
    top_indices=similar_indices[:top_n]
    recommendations=df.iloc[top_indices][
        ["name","cuisines","location","rate"]
    ].copy()
    recommendations["similarity_score"]=similarity_scores[top_indices]
    recommendations.reset_index(drop=True)
    return recommendations

In [15]:
df["name"].sample(20)

19057                   Vijaya Gardenia
19359                         Abhiruchi
30871                           Sunehri
12646                       Kaka-T-Cafe
36632                 Sri Krishna Sagar
37796                   Behrouz Biryani
19928                       Burger King
25228              Ambur Biriyani Point
9707                      Lazeez Xpress
6156                    Gelato Italiano
6943                           1000 B.C
36970                      Antarastriya
19063                      Juice Fresco
16012       New Chickpete Donne Biryani
18679                         The Airos
12806    Savoury - Sea Shell Restaurant
13816                          Truffles
8635       Myu Bar at Gilly's Redefined
7923                        Tandoor Hut
13052                   Bengaluru House
Name: name, dtype: str

In [18]:
recommend_restaurants("Jalsa")

,name,cuisines,location,rate,similarity_score
16882,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,1.000000
2384,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,1.000000
1941,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,1.000000
400,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,1.000000
533,Food Box Cafe,"mughlai, north indian, chinese",banashankari,3.6,1.000000
485,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,1.000000
15333,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,1.000000
16241,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,1.000000
2774,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,1.000000
527,J Spice,"north indian, chinese",banashankari,3.8,0.839312
